# Faster R-CNN 학습 + 평가 (torchvision, ResNet50 FPN v2)

**환경**: Colab A100  
**모델**: `fasterrcnn_resnet50_fpn_v2` (COCO pretrained, 43.6M)  
**조건**: epochs=15, batch=16, imgsz=640, seed=42  
**Optimizer**: SGD lr=0.005, momentum=0.9, wd=0.0005, StepLR(step=5, gamma=0.1)  
**증강**: CLAHE / GaussianBlur / MotionBlur (ablation Top-3) + letterbox

## 노트북 구조
- **학습 파트**: 환경 → 데이터 → 모델 → 학습 → 곡선 → 백업
- **평가 파트**: 환경 셋업 (자체 완결) → 33장 평가 → JSON 저장


## 셀 1 — 환경 점검


In [ ]:
import torch, sys, platform
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')

# 시드 고정 (전역)
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'\nseed={SEED} 고정')


## 셀 2 — 패키지 설치


In [ ]:
!pip install -q albumentations==1.4.18 supervision==0.25.0
print('패키지 설치 완료')


## 셀 3 — Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 셀 4 — 학습 데이터 (dataset_yolo) 준비


In [ ]:
import os, time, subprocess
from pathlib import Path

DATA_ZIP   = '/content/drive/MyDrive/dataset_yolo.zip'
DATA_ROOT  = Path('/content/dataset_yolo')

# 이미 풀려있으면 스킵
if (DATA_ROOT / 'images' / 'train').exists() and len(list((DATA_ROOT / 'images' / 'train').iterdir())) > 0:
    print(f'스킵: {DATA_ROOT} 이미 존재')
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    # zip 구조 먼저 확인
    print('zip 구조 확인 중...')
    !unzip -l {DATA_ZIP} | head -20
    print()

    t0 = time.time()
    print(f'압축 풀기 시작: {DATA_ZIP}')
    !unzip -q {DATA_ZIP} -d /content/
    print(f'완료: {(time.time()-t0)/60:.1f}분')

# 검증
for split in ['train', 'val']:
    img_n = len(list((DATA_ROOT / 'images' / split).glob('*.*')))
    lbl_n = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split}: 이미지 {img_n:,} / 라벨 {lbl_n:,}')


import zipfile, shutil, json
from pathlib import Path

EVAL_ZIP = '/content/drive/MyDrive/eval_package.zip'
EVAL_ROOT = Path('/content/eval_package')

# 깨진 거 정리
if EVAL_ROOT.exists():
    shutil.rmtree(EVAL_ROOT)

# zipfile 로 풀기 (try-except 없이 바로 추출)
with zipfile.ZipFile(EVAL_ZIP, 'r') as z:
    z.extractall(EVAL_ROOT)

# 검증
imgs = sorted(EVAL_ROOT.rglob('*.png'))
print(f'평가 이미지: {len(imgs)}장')
print('\n파일명 (한글 정상 확인):')
for p in imgs[:5]:
    print(f'  {p.name}')

# answer_key 매칭
with open(EVAL_ROOT / 'answer_key.json', 'r', encoding='utf-8') as f:
    ANSWER_KEY = json.load(f)

img_names = {p.name for p in imgs}
ak_names = set(ANSWER_KEY.keys())
matched = img_names & ak_names

print(f'\n매칭: {len(matched)}/33')
unmatched_img = img_names - ak_names
unmatched_ak = ak_names - img_names
if unmatched_img:
    print(f'⚠️ 디스크에만 있음: {len(unmatched_img)}개')
    for x in list(unmatched_img)[:5]: print(f'   {x}')
if unmatched_ak:
    print(f'⚠️ answer_key에만 있음: {len(unmatched_ak)}개')
    for x in list(unmatched_ak)[:5]: print(f'   {x}')

## 셀 5 — 평가셋 준비


In [ ]:
import zipfile, shutil, json
from pathlib import Path

# ── 데이터 경로 정정 ──
DATA_ROOT = Path('/content')   # zip 평면 구조 대응
EVAL_ROOT = Path('/content/eval_package')

# ── 평가셋 zipfile로 풀기 (한글 정상) ──
EVAL_ZIP = '/content/drive/MyDrive/eval_package.zip'
if EVAL_ROOT.exists():
    shutil.rmtree(EVAL_ROOT)
with zipfile.ZipFile(EVAL_ZIP, 'r') as z:
    z.extractall(EVAL_ROOT)

# ── answer_key 로드 ──
with open(EVAL_ROOT / 'answer_key.json', 'r', encoding='utf-8') as f:
    ANSWER_KEY = json.load(f)

# ── 검증 ──
for split in ['train', 'val']:
    img_n = len(list((DATA_ROOT / 'images' / split).glob('*.*')))
    lbl_n = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split}: 이미지 {img_n:,} / 라벨 {lbl_n:,}')

imgs = sorted(EVAL_ROOT.rglob('*.png'))
img_names = {p.name for p in imgs}
matched = img_names & set(ANSWER_KEY.keys())
print(f'\n평가 이미지: {len(imgs)}장')
print(f'answer_key: {len(ANSWER_KEY)}개')
print(f'매칭: {len(matched)}/33')
print(f'pill.yaml: {(DATA_ROOT / "pill.yaml").exists()}')

EVAL_ZIP  = '/content/drive/MyDrive/pill_detection/eval_package.zip'
EVAL_ROOT = Path('/content/eval_package')

if (EVAL_ROOT / 'test_images').exists() and len(list((EVAL_ROOT / 'test_images').glob('*.png'))) > 0:
    print(f'스킵: {EVAL_ROOT} 이미 존재')
else:
    EVAL_ROOT.mkdir(parents=True, exist_ok=True)
    !unzip -o -q {EVAL_ZIP} -d {EVAL_ROOT}

# 검증
imgs = sorted(EVAL_ROOT.rglob('*.png'))
print(f'평가 이미지: {len(imgs)}장')
ans_key = list(EVAL_ROOT.rglob('answer_key.json'))
print(f'answer_key: {ans_key}')

import json
with open(ans_key[0], 'r', encoding='utf-8') as f:
    ANSWER_KEY = json.load(f)
print(f'answer_key 항목 수: {len(ANSWER_KEY)}')
print('샘플:', list(ANSWER_KEY.items())[0])


from pathlib import Path
import json

# 데이터 경로 재설정
DATA_ROOT = Path('/content')   # zip이 평면이라 dataset_yolo/ 아닌 /content/

# 검증
for split in ['train', 'val']:
    img_n = len(list((DATA_ROOT / 'images' / split).glob('*.*')))
    lbl_n = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split}: 이미지 {img_n:,} / 라벨 {lbl_n:,}')

# 평가셋 경로 재설정
EVAL_ROOT = Path('/content/eval_package')
imgs = sorted(EVAL_ROOT.rglob('*.png'))
print(f'\n평가 이미지: {len(imgs)}장')

ans_key = list(EVAL_ROOT.rglob('answer_key.json'))
print(f'answer_key 파일: {ans_key}')

if ans_key:
    with open(ans_key[0], 'r', encoding='utf-8') as f:
        ANSWER_KEY = json.load(f)
    print(f'answer_key 항목 수: {len(ANSWER_KEY)}')
    print('샘플:', list(ANSWER_KEY.items())[0])
else:
    print('⚠️ answer_key.json 없음 — eval_package 내부 확인 필요')
    !ls /content/eval_package/

## 셀 6 — 클래스 정보 로드


In [ ]:
import yaml

with open(DATA_ROOT / 'pill.yaml', 'r', encoding='utf-8') as f:
    pill_yaml = yaml.safe_load(f)

names = pill_yaml['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]

CLASS_NAMES = list(names)  # 0-indexed
NUM_CLASSES = len(CLASS_NAMES)
print(f'클래스 수: {NUM_CLASSES}')
print(f'샘플 클래스: {CLASS_NAMES[:5]}')

# 학습 시 라벨 인덱스: torchvision은 1-indexed (0=background)
# YOLO label txt 의 class_idx → +1 해서 사용


from pathlib import Path

# 데이터 경로 정정
DATA_ROOT = Path('/content')             # zip이 평면 구조라 /content/ 바로 아래
EVAL_ROOT = Path('/content/eval_package')

# 검증
for split in ['train', 'val']:
    img_n = len(list((DATA_ROOT / 'images' / split).glob('*.*')))
    lbl_n = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split}: 이미지 {img_n:,} / 라벨 {lbl_n:,}')

print(f'평가셋: {len(list(EVAL_ROOT.rglob("*.png")))}장')
print(f'pill.yaml: {(DATA_ROOT / "pill.yaml").exists()}')

## 셀 7 — Dataset 클래스 정의


In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path


class YoloDetDataset(Dataset):
    def __init__(self, root: Path, split: str, transform=None):
        self.img_dir = Path(root) / 'images' / split
        self.lbl_dir = Path(root) / 'labels' / split
        self.images = sorted([p for p in self.img_dir.glob('*.*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # YOLO 라벨 읽기 (normalized cxcywh)
        lbl_path = self.lbl_dir / f'{img_path.stem}.txt'
        boxes_xyxy = []
        labels = []
        if lbl_path.exists():
            with open(lbl_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts)
                    # normalized cxcywh → absolute xyxy
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    x2 = (cx + bw/2) * w
                    y2 = (cy + bh/2) * h
                    # 경계 클립 (안전)
                    x1 = max(0.0, min(x1, w-1))
                    y1 = max(0.0, min(y1, h-1))
                    x2 = max(0.0, min(x2, w-1))
                    y2 = max(0.0, min(y2, h-1))
                    if x2 > x1 and y2 > y1:
                        boxes_xyxy.append([x1, y1, x2, y2])
                        labels.append(int(cls) + 1)   # +1 (background=0 예약)

        # transform 적용
        if self.transform is not None:
            transformed = self.transform(image=img, bboxes=boxes_xyxy, labels=labels)
            img = transformed['image']
            boxes_xyxy = transformed['bboxes']
            labels = transformed['labels']

        # tensor 변환 (transform 안에서 ToTensorV2 처리)
        if len(boxes_xyxy) > 0:
            boxes_t = torch.as_tensor(boxes_xyxy, dtype=torch.float32)
            labels_t = torch.as_tensor(labels, dtype=torch.int64)
        else:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)

        target = {
            'boxes': boxes_t,
            'labels': labels_t,
            'image_id': torch.tensor([idx]),
        }
        return img, target


print('YoloDetDataset 정의 완료')


## 셀 8 — 증강 + 정규화


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMGSZ = 640

train_transform = A.Compose([
    # 증강 Top-3 (설계서 v4)
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
    A.GaussianBlur(blur_limit=7, p=0.3),
    A.MotionBlur(blur_limit=7, p=0.3),
    # letterbox
    A.LongestMaxSize(max_size=IMGSZ),
    A.PadIfNeeded(min_height=IMGSZ, min_width=IMGSZ, border_mode=cv2.BORDER_CONSTANT, value=(0,0,0)),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

val_transform = A.Compose([
    A.LongestMaxSize(max_size=IMGSZ),
    A.PadIfNeeded(min_height=IMGSZ, min_width=IMGSZ, border_mode=cv2.BORDER_CONSTANT, value=(0,0,0)),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

# ToTensorV2는 [H,W,C] uint8 → [C,H,W] uint8 출력. Faster R-CNN은 0~1 float을 기대.
# → DataLoader 단계에서 / 255 처리 (collate_fn 안에)

print('train_transform / val_transform 정의 완료')


## 셀 9 — DataLoader


In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images, targets = zip(*batch)
    # uint8 → float [0,1]
    images = [img.float() / 255.0 for img in images]
    return list(images), list(targets)


train_ds = YoloDetDataset(DATA_ROOT, 'train', transform=train_transform)
val_ds   = YoloDetDataset(DATA_ROOT, 'val',   transform=val_transform)
print(f'train: {len(train_ds):,} / val: {len(val_ds):,}')

BATCH = 16

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=4, collate_fn=collate_fn, pin_memory=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                          num_workers=4, collate_fn=collate_fn, pin_memory=True,
                          persistent_workers=True)

# sanity check 1배치
images, targets = next(iter(train_loader))
print(f'\nbatch 크기: {len(images)}')
print(f'이미지 shape: {images[0].shape} (expect [3, 640, 640])')
print(f'타겟[0] boxes shape: {targets[0]["boxes"].shape}')
print(f'타겟[0] labels (앞 5개): {targets[0]["labels"][:5].tolist()}')
print(f'라벨 범위: {min(t["labels"].min().item() for t in targets if len(t["labels"])>0)} ~ '
      f'{max(t["labels"].max().item() for t in targets if len(t["labels"])>0)} (expect 1~118)')


## 셀 10 — Faster R-CNN ResNet50 FPN v2 모델


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# pretrained 로드
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights)

# head 교체 (background 포함 → +1)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)

# 디바이스
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 파라미터 수
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터: {total_params/1e6:.1f}M')
print(f'학습 가능: {trainable_params/1e6:.1f}M')
print(f'device: {device}')


## 셀 11 — 학습 루프 (epochs=15)

A100 기준 약 80분 소요. best/last 체크포인트는 OUT_DIR에 저장.


In [ ]:
import time, math
from pathlib import Path

EPOCHS = 15
EXP_NAME = 'exp_faster_rcnn'
OUT_DIR = Path('/content/runs') / EXP_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optimizer (torchvision Faster R-CNN tutorial 표준)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# 로그
import json
log = {'train_loss': [], 'val_loss': [], 'lr': [], 'epoch_time_s': []}

def train_one_epoch(model, optimizer, loader, device, epoch, total_epochs):
    model.train()
    losses_sum = 0.0
    n_batches = len(loader)
    t0 = time.time()
    for i, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        if not math.isfinite(loss.item()):
            print(f'⚠️  loss NaN/Inf at batch {i}, 스킵')
            continue

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses_sum += loss.item()
        if (i + 1) % 50 == 0 or i == n_batches - 1:
            elapsed = time.time() - t0
            print(f'  [{epoch}/{total_epochs}] batch {i+1}/{n_batches}  '
                  f'loss={losses_sum/(i+1):.4f}  ({elapsed:.0f}s elapsed)')
    return losses_sum / max(n_batches, 1)


@torch.no_grad()
def validate(model, loader, device):
    # torchvision Faster R-CNN: eval()에선 prediction만 리턴, loss 안 줌.
    # → loss 측정 위해 train()모드 + no_grad 조합 (gradient 안 흐름)
    model.train()
    losses_sum = 0.0
    n_batches = len(loader)
    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        losses_sum += loss.item()
    return losses_sum / max(n_batches, 1)


best_val = float('inf')
total_t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    print(f'\n=== epoch {epoch}/{EPOCHS} ===')
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch, EPOCHS)
    val_loss   = validate(model, val_loader, device)
    lr_scheduler.step()

    epoch_t = time.time() - t0
    cur_lr = optimizer.param_groups[0]['lr']

    log['train_loss'].append(train_loss)
    log['val_loss'].append(val_loss)
    log['lr'].append(cur_lr)
    log['epoch_time_s'].append(epoch_t)

    print(f'  train={train_loss:.4f}  val={val_loss:.4f}  lr={cur_lr:.5f}  ({epoch_t:.0f}s)')

    # best 저장
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), OUT_DIR / 'best.pt')
        print(f'  ✅ best 갱신 → {OUT_DIR / "best.pt"}')

    # last 저장
    torch.save(model.state_dict(), OUT_DIR / 'last.pt')

# 로그 저장
with open(OUT_DIR / 'log.json', 'w', encoding='utf-8') as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

total_min = (time.time() - total_t0) / 60
print(f'\n학습 완료. 총 소요: {total_min:.1f}분 / best val_loss: {best_val:.4f}')
print(f'산출물 위치: {OUT_DIR}')


## 셀 12 — 학습 곡선


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for n in ['NanumGothic', 'Malgun Gothic', 'AppleGothic']:
    if n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = n; break
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ep = list(range(1, len(log['train_loss']) + 1))

axes[0].plot(ep, log['train_loss'], 'o-', label='train', color='#3498db')
axes[0].plot(ep, log['val_loss'], 's-', label='val', color='#e67e22')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, log['lr'], 'd-', color='#2ecc71')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('learning rate')
axes[1].set_title('LR Schedule')
axes[1].grid(alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'저장: {OUT_DIR / "training_curve.png"}')


## 셀 13 — Drive 백업 (학습 산출물)


In [ ]:
import shutil

DRIVE_BACKUP = Path('/content/drive/MyDrive/pill_detection/results') / EXP_NAME
DRIVE_BACKUP.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_BACKUP.exists():
    print(f'⚠️ 기존 백업 존재: {DRIVE_BACKUP}')
    print('  덮어쓰려면 수동으로 셀 수정 필요')
else:
    shutil.copytree(OUT_DIR, DRIVE_BACKUP)
    print(f'✅ 백업 완료: {DRIVE_BACKUP}')

# 폴더 안 파일 목록
for f in sorted(DRIVE_BACKUP.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')


---

## 셀 14 — 평가 환경 셋업 (자체 완결, 런타임 끊겼을 경우 여기부터 시작)

best.pt 로드 + eval_package + 헬퍼 함수까지 한 번에. 학습 직후 이어서 돌려도 OK, 단독 실행도 OK.


In [ ]:
# 평가 환경 셋업 (자체 완결: 런타임 끊겼을 경우 여기부터 시작 가능)
import os, json, zipfile, shutil, yaml, torch
import numpy as np
from pathlib import Path
from PIL import Image
import cv2 as _cv2

# === 1) Drive 마운트 (안 됐을 경우만) ===
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# === 2) 학습 시 사용한 파라미터 (변수 복원용) ===
EXP_NAME_EVAL = 'exp_faster_rcnn'
AUTHOR        = '승원'
BATCH = 16; IMGSZ = 640; EPOCHS = 15; SEED = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === 3) eval_package 풀기 (한글 파일명 보존을 위해 zipfile 사용) ===
EVAL_ZIP = '/content/drive/MyDrive/pill_detection/eval_package.zip'
EVAL_ROOT = Path('/content/eval_package')
if EVAL_ROOT.exists():
    shutil.rmtree(EVAL_ROOT)
with zipfile.ZipFile(EVAL_ZIP, 'r') as z:
    z.extractall(EVAL_ROOT)

ANSWER_KEY = json.load(open(EVAL_ROOT / 'answer_key.json', encoding='utf-8'))
test_imgs = sorted(EVAL_ROOT.rglob('*.png'))
matched = {p.name for p in test_imgs} & set(ANSWER_KEY.keys())
print(f'평가 이미지: {len(test_imgs)}장 / 매칭: {len(matched)}/33')

# === 4) CLASS_NAMES (pill.yaml에서) ===
PILL_YAML_CANDIDATES = ['/content/pill.yaml', '/content/dataset_yolo/pill.yaml']
pill_yaml_path = next((p for p in PILL_YAML_CANDIDATES if os.path.exists(p)), None)
if pill_yaml_path is None:
    # dataset_yolo.zip에서 추출
    with zipfile.ZipFile('/content/drive/MyDrive/dataset_yolo.zip', 'r') as z:
        yaml_names = [n for n in z.namelist() if n.endswith('pill.yaml')]
        if yaml_names:
            z.extract(yaml_names[0], '/content/')
            extracted = Path('/content') / yaml_names[0]
            pill_yaml_path = '/content/pill.yaml'
            if extracted != Path(pill_yaml_path):
                shutil.copy2(extracted, pill_yaml_path)

with open(pill_yaml_path, 'r', encoding='utf-8') as f:
    pill_yaml = yaml.safe_load(f)
names = pill_yaml['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]
CLASS_NAMES = list(names)
NUM_CLASSES = len(CLASS_NAMES)
print(f'클래스 수: {NUM_CLASSES}')

# === 5) best.pt 로드 + 평가 모델 ===
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

BEST_PT = Path('/content/drive/MyDrive/pill_detection/results') / EXP_NAME_EVAL / 'best.pt'
print(f'\nbest.pt: {BEST_PT}  exists={BEST_PT.exists()}')

model = fasterrcnn_resnet50_fpn_v2(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)
model.to(device)
model.load_state_dict(torch.load(BEST_PT, map_location=device))
model.eval()

# Top-3 추출 위해 추론 후처리 완화
model.roi_heads.score_thresh        = 0.01
model.roi_heads.nms_thresh          = 0.7
model.roi_heads.detections_per_img  = 300
print('✅ 평가 모델 준비 완료 (score_thresh=0.01, nms=0.7, det_per_img=300)')

# === 6) 헬퍼: letterbox 역변환 + IoU 그룹핑 + Top-3 ===
def letterbox_inverse(box_xyxy, orig_w, orig_h, target=640):
    ratio = target / max(orig_w, orig_h)
    new_w = orig_w * ratio
    new_h = orig_h * ratio
    pad_x = (target - new_w) / 2
    pad_y = (target - new_h) / 2
    x1, y1, x2, y2 = box_xyxy
    x1 = (x1 - pad_x) / ratio; y1 = (y1 - pad_y) / ratio
    x2 = (x2 - pad_x) / ratio; y2 = (y2 - pad_y) / ratio
    x1 = max(0.0, min(x1, orig_w)); y1 = max(0.0, min(y1, orig_h))
    x2 = max(0.0, min(x2, orig_w)); y2 = max(0.0, min(y2, orig_h))
    return [x1, y1, x2, y2]

def compute_iou(a, b):
    xA = max(a[0], b[0]); yA = max(a[1], b[1])
    xB = min(a[2], b[2]); yB = min(a[3], b[3])
    inter = max(0.0, xB - xA) * max(0.0, yB - yA)
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter
    return inter / union if union > 1e-9 else 0.0

def group_and_top3(boxes_orig, labels_1based, scores, iou_thr=0.5, max_top=3):
    if len(boxes_orig) == 0:
        return []
    order = np.argsort(-np.array(scores))
    bs = [boxes_orig[i] for i in order]
    ls = [int(labels_1based[i]) for i in order]
    ss = [float(scores[i]) for i in order]
    used = [False] * len(bs)
    detections = []
    for i in range(len(bs)):
        if used[i]: continue
        group_idx = [i]; used[i] = True
        for j in range(i + 1, len(bs)):
            if used[j]: continue
            if compute_iou(bs[i], bs[j]) > iou_thr:
                group_idx.append(j); used[j] = True
        cls_best = {}
        for k in group_idx:
            cls = ls[k]
            if (cls not in cls_best) or (ss[k] > cls_best[cls][0]):
                cls_best[cls] = (ss[k], k)
        top = sorted(cls_best.items(), key=lambda x: -x[1][0])[:max_top]
        rep_box = bs[top[0][1][1]]
        detections.append({
            'bbox': [float(x) for x in rep_box],
            'top3_class_idx':   [int(c - 1) for c, _ in top],
            'top3_class_name':  [CLASS_NAMES[c - 1] for c, _ in top],
            'top3_confidence':  [float(s) for _, (s, _) in top],
        })
    return detections

def predict_top3(image_path, model, device, imgsz=640):
    img_bgr = _cv2.imread(str(image_path))
    img_rgb = _cv2.cvtColor(img_bgr, _cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    ratio = imgsz / max(W, H)
    new_w, new_h = int(W * ratio), int(H * ratio)
    resized = _cv2.resize(img_rgb, (new_w, new_h))
    pad_x = (imgsz - new_w) // 2; pad_y = (imgsz - new_h) // 2
    canvas = np.zeros((imgsz, imgsz, 3), dtype=np.uint8)
    canvas[pad_y:pad_y+new_h, pad_x:pad_x+new_w] = resized
    x = torch.from_numpy(canvas).permute(2, 0, 1).float() / 255.0
    x = x.to(device)
    with torch.no_grad():
        out = model([x])[0]
    boxes  = out['boxes'].cpu().numpy().tolist()
    labels = out['labels'].cpu().numpy().tolist()
    scores = out['scores'].cpu().numpy().tolist()
    boxes_orig = [letterbox_inverse(b, W, H, imgsz) for b in boxes]
    return group_and_top3(boxes_orig, labels, scores, iou_thr=0.5, max_top=3), W, H

print('\n✅ 헬퍼 함수 정의 완료 (letterbox_inverse / compute_iou / group_and_top3 / predict_top3)')


## 셀 15 — 33장 평가 (Top-1 / Top-3)


In [ ]:
import time, json
from pathlib import Path

test_imgs = sorted(EVAL_ROOT.rglob('*.png'))
print(f'평가 이미지: {len(test_imgs)}장')

results = []
top1_hit = 0
top3_hit = 0
detected_n = 0
n = len(test_imgs)

t0 = time.time()
for img_path in test_imgs:
    fname = img_path.name
    if fname not in ANSWER_KEY:
        print(f'⚠️ answer_key 없음: {fname}')
        continue

    gt = ANSWER_KEY[fname]
    gt_dl_name  = gt['dl_name']
    gt_class_idx = gt['class_idx']

    detections, W, H = predict_top3(img_path, model, device, imgsz=640)

    if len(detections) > 0:
        detected_n += 1
        top1_names = [d['top3_class_name'][0] for d in detections if d.get('top3_class_name')]
        top3_names = [nm for d in detections for nm in d.get('top3_class_name', [])]

        if gt_dl_name in top1_names:
            top1_hit += 1
        if gt_dl_name in top3_names:
            top3_hit += 1

    results.append({
        'filename': fname,
        'img_width': W,
        'img_height': H,
        'gt_dl_name': gt_dl_name,
        'gt_class_idx': gt_class_idx,
        'num_detections': len(detections),
        'detections': detections,
    })

elapsed = time.time() - t0
print(f'\n평가 완료: {elapsed:.1f}초 ({elapsed/n:.2f}s/img)')
print(f'Detection rate: {detected_n}/{n} = {detected_n/n:.3f}')
print(f'Top-1 accuracy: {top1_hit}/{n}    = {top1_hit/n:.3f}')
print(f'Top-3 accuracy: {top3_hit}/{n}    = {top3_hit/n:.3f}')

## 셀 16 — JSON 저장 + Drive 백업

`_model_collection/` 폴더에 백업 → 로컬 노트북(compare_experiments)에서 다른 모델과 비교


In [ ]:
import json, shutil
from datetime import datetime
from pathlib import Path

total_params_M = sum(p.numel() for p in model.parameters()) / 1e6

output_data = {
    'exp_name': 'exp_faster_rcnn',
    'author': '승원',
    'experiment_type': 'model',
    'model_framework': 'torchvision',
    'model_name': 'fasterrcnn_resnet50_fpn_v2',
    'model_params_M': round(total_params_M, 1),
    'batch_size_used': 16,
    'imgsz': 640,
    'epochs': 15,
    'seed': 42,
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'num_images': n,
    'detection_rate': round(detected_n / n, 4),
    'top1_accuracy':  round(top1_hit / n, 4),
    'top3_accuracy':  round(top3_hit / n, 4),
    'results': results,
}

# 1) 로컬 저장
EVAL_OUT = Path('/content/runs/exp_faster_rcnn/eval')
EVAL_OUT.mkdir(parents=True, exist_ok=True)
JSON_PATH = EVAL_OUT / 'exp_faster_rcnn_results.json'
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)
print(f'✅ 로컬 저장: {JSON_PATH}')

# 2) Drive 모델 비교용 폴더에 백업
DRIVE_COLLECTION = Path('/content/drive/MyDrive/pill_detection/results/_model_collection')
DRIVE_COLLECTION.mkdir(parents=True, exist_ok=True)
DRIVE_JSON = DRIVE_COLLECTION / 'exp_faster_rcnn_results.json'
shutil.copy2(JSON_PATH, DRIVE_JSON)
print(f'✅ Drive 백업: {DRIVE_JSON}')

# 3) 요약
print()
print('=' * 60)
print(f'  실험명: exp_faster_rcnn')
print(f'  모델  : fasterrcnn_resnet50_fpn_v2 ({total_params_M:.1f}M)')
print(f'  Detection rate: {detected_n}/{n} = {detected_n/n:.3f}')
print(f'  Top-1 accuracy: {top1_hit}/{n}    = {top1_hit/n:.3f}')
print(f'  Top-3 accuracy: {top3_hit}/{n}    = {top3_hit/n:.3f}')
print('=' * 60)